## 24. Todo 跟踪：读出 agent 的任务清单

> 来源：[Todo Lists](https://code.claude.com/docs/en/agent-sdk/todo-tracking)

SDK 没有单独的"任务进度 API"：agent 的任务清单靠**工具调用**维护，进度信息全部走消息流。监听 `AssistantMessage` 里对应的 `ToolUseBlock`（§7「核心技能：读懂消息流」），就能把清单实时渲染成自己的进度 UI。

- **生命周期**：任务被识别时以 `pending` 创建 → 开工转 `in_progress` → 完成转 `completed` → 整组任务全部完成后清单移除。
- **什么时候会建清单**：需要 3 个以上不同动作的多步任务、用户一次给了多项任务、值得展示进度的非平凡操作、用户显式要求用 todo 组织。很短的单步请求可能直接跳过。

### 24.1 两套工具：Task 系（🆕 默认）与 TodoWrite

任务清单工具的换代按 **CLI 版本**算，不看 Python 包版本：Python SDK 的行为跟随它启动的那份 Claude Code CLI（pip 包内捆绑的，或 `cli_path` 指向的）。CLI ≥ v2.1.142 起默认用结构化的 Task 系工具（`TaskCreate` / `TaskUpdate` / `TaskGet` / `TaskList`）取代单一的 `TodoWrite`；还想收 `TodoWrite` 就设 `env={"CLAUDE_CODE_ENABLE_TASKS": "0"}`。

| | `TodoWrite` | Task 系工具 |
|---|---|---|
| 更新方式 | 一次工具调用重写整个 `todos` 数组 | `TaskCreate` 加一条，`TaskUpdate` 按 `taskId` 改一条 |
| 监听匹配 | `block.name == "TodoWrite"` | `block.name` 为 `"TaskCreate"` / `"TaskUpdate"` |
| 条目结构 | `{content, status, activeForm}` | `TaskCreate` 入参 `{subject, description, activeForm?, metadata?}`；`TaskUpdate` 入参 `{taskId, status?, subject?, description?, activeForm?, addBlocks?, addBlockedBy?, owner?, metadata?}` |
| 状态取值 | `pending` / `in_progress` / `completed` | 同左，另有 `status: "deleted"` 表示删除该条 |
| 渲染方式 | 直接渲染 `block.input["todos"]` 全表 | 跨调用自己累积成 map，或等 `TaskList` 的 tool result 拿整表快照 |

### 24.2 监听 Task 系（现行默认）

两个坑直接决定监听代码怎么写：

> [!warning] task ID 不在 `TaskCreate` 的入参里
> 分配的 ID 在对应 **tool_result** 的 `{"task": {"id": ..., "subject": ...}}` 里，只读 `ToolUseBlock` 拿不到新任务的 ID。要维护按 ID 索引的 map，就得同时监听 tool result 抓 ID。

> [!warning] 流里的 `tool_use` input 是模型原始输出，键名修复不回写
> Claude Code 执行前会把 `id`/`task_id` 修正为 `taskId`、`active_form` 修正为 `activeForm`，但流里看到的仍是修复前的写法。监听代码要防御性读键名：`input.get("taskId") or input.get("id") or input.get("task_id")`。

下方示例用单次 `query()`：触 `max_turns` 上限时先 yield `error_max_turns` 的 `ResultMessage` 再抛异常，所以循环要包 try（§4「两个入口与两种输入模式」的错误语义）。

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ToolUseBlock


async def monitor_task_tools():
    try:
        async for message in query(
            prompt="Optimize my React app performance and track progress with todos",
            options=ClaudeAgentOptions(max_turns=15),
        ):
            if not isinstance(message, AssistantMessage):
                continue
            for block in message.content:
                if not isinstance(block, ToolUseBlock):
                    continue
                if block.name == "TaskCreate":
                    print(f"+ {block.input['subject']}")
                elif block.name == "TaskUpdate" and block.input.get("status"):
                    # 防御性读键名：流里可能是修复前的 id/task_id
                    task_id = (
                        block.input.get("taskId")
                        or block.input.get("id")
                        or block.input.get("task_id")
                    )
                    if task_id:
                        print(f"  {task_id} -> {block.input['status']}")
    except Exception as error:
        # 单次 query() 在 yield 错误 result 之后还会抛异常（§4）
        print(f"Session ended with an error: {error}")


await monitor_task_tools()

### 24.3 监听 TodoWrite（旧开关）

设 `CLAUDE_CODE_ENABLE_TASKS=0` 后 `TodoWrite` 恢复出现，每次调用携带**整张清单**，本地状态直接覆盖即可。渲染细节：条目处于 `in_progress` 时显示 `activeForm`（进行时文案），其余状态显示 `content`。

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ToolUseBlock


class TodoTracker:
    """跟踪 TodoWrite 全量清单并渲染进度。"""

    def __init__(self):
        self.todos = []

    def display_progress(self):
        if not self.todos:
            return
        completed = len([t for t in self.todos if t["status"] == "completed"])
        print(f"\nProgress: {completed}/{len(self.todos)} completed")
        for i, todo in enumerate(self.todos):
            icon = {"completed": "✅", "in_progress": "🔧"}.get(todo["status"], "⬜")
            # in_progress 显示进行时文案 activeForm，其余显示 content
            text = todo["activeForm"] if todo["status"] == "in_progress" else todo["content"]
            print(f"{i + 1}. {icon} {text}")

    async def track(self, prompt):
        try:
            async for message in query(
                prompt=prompt,
                options=ClaudeAgentOptions(
                    max_turns=20,
                    env={"CLAUDE_CODE_ENABLE_TASKS": "0"},  # 重新启用 TodoWrite
                ),
            ):
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, ToolUseBlock) and block.name == "TodoWrite":
                            self.todos = block.input["todos"]  # 每次都是整表，直接覆盖
                            self.display_progress()
        except Exception as error:
            print(f"Session ended with an error: {error}")


await TodoTracker().track("Build a complete authentication system with todos")